In [34]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit

In [35]:
def make_cv_folds(train_pool, dates, n_splits=5):
    for fold, (train_idx, val_idx) in enumerate(TimeSeriesSplit(n_splits=n_splits).split(dates)):
        train_dates = dates[train_idx]# gives the training dates for that specific fold like what the actual dates are
        val_dates = dates[val_idx]# gives the validation dates for that specific fold like what the actual validation dates are
        assert not set(train_dates) & set(val_dates)
        train_mask = train_pool['FlightDate'].isin(train_dates)# finds the rows that correspond to the dates and returns a boolean array
        val_mask = train_pool['FlightDate'].isin(val_dates)# finds the rows that correspond to the dates and returns a boolean array
        train_fold = train_pool[train_mask]#gets the actual flights rows that belong to the training dates
        val_fold = train_pool[val_mask]# gets the actual flight rows that belong to the validation dates
        yield fold, train_fold, val_fold, train_dates, val_dates

In [36]:
df = pd.read_csv('../data/interim/seattle_ontime_clean.csv') #Loading the csv
df.shape
df.columns


Index(['Unnamed: 0', 'Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek',
       'FlightDate', 'Reporting_Airline', 'DOT_ID_Reporting_Airline',
       'IATA_CODE_Reporting_Airline',
       ...
       'Div4TailNum', 'Div5Airport', 'Div5AirportID', 'Div5AirportSeqID',
       'Div5WheelsOn', 'Div5TotalGTime', 'Div5LongestGTime', 'Div5WheelsOff',
       'Div5TailNum', 'Unnamed: 109'],
      dtype='object', length=111)

In [37]:

df['FlightDate'] = pd.to_datetime(df["FlightDate"])
df['FlightDate'].head()

0   2024-08-01
1   2024-08-02
2   2024-08-03
3   2024-08-04
4   2024-08-01
Name: FlightDate, dtype: datetime64[ns]

In [38]:
post_flight = [
    'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups',
    'TaxiOut', 'WheelsOff', 'WheelsOn', 'TaxiIn', 'ArrTime', 'ArrDelayMinutes',
    'ArrDel15', 'ArrivalDelayGroups', 'ActualElapsedTime', 'AirTime',
    'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay',
    'FirstDepTime', 'TotalAddGTime', 'LongestAddGTime',
    'Cancelled', 'CancellationCode', 'Diverted',
    'DivAirportLandings', 'DivReachedDest', 'DivActualElapsedTime', 'DivArrDelay',
    'DivDistance',
    'Div1Airport', 'Div1AirportID', 'Div1AirportSeqID', 'Div1WheelsOn',
    'Div1TotalGTime', 'Div1LongestGTime', 'Div1WheelsOff', 'Div1TailNum',
    'Div2Airport', 'Div2AirportID', 'Div2AirportSeqID', 'Div2WheelsOn',
    'Div2TotalGTime', 'Div2LongestGTime', 'Div2WheelsOff', 'Div2TailNum',
    'Div3Airport', 'Div3AirportID', 'Div3AirportSeqID', 'Div3WheelsOn',
    'Div3TotalGTime', 'Div3LongestGTime', 'Div3WheelsOff', 'Div3TailNum',
    'Div4Airport', 'Div4AirportID', 'Div4AirportSeqID', 'Div4WheelsOn',
    'Div4TotalGTime', 'Div4LongestGTime', 'Div4WheelsOff', 'Div4TailNum',
    'Div5Airport', 'Div5AirportID', 'Div5AirportSeqID', 'Div5WheelsOn',
    'Div5TotalGTime', 'Div5LongestGTime', 'Div5WheelsOff', 'Div5TailNum',
]
drop_cols = [
    'Reporting_Airline', 'DOT_ID_Reporting_Airline', 'Tail_Number',
    'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID',
    'OriginCityName', 'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac',
    'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID',
    'DestCityName', 'DestState', 'DestStateFips', 'DestStateName', 'DestWac',
    'DepTimeBlk', 'ArrTimeBlk', 'Flights', 'DistanceGroup','Origin','Unnamed: 0', 'Unnamed: 109'
]


df = df.drop(columns=post_flight + drop_cols)
df.columns

Index(['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate',
       'IATA_CODE_Reporting_Airline', 'Flight_Number_Reporting_Airline',
       'Dest', 'CRSDepTime', 'CRSArrTime', 'ArrDelay', 'CRSElapsedTime',
       'Distance'],
      dtype='object')

## Why post-flight and redundant columns were dropped

Post-flight features like `DepDelay`, `DepTime`, `TaxiOut`, and `ActualElapsedTime` were dropped because they don't exist when someone is booking a flight. If we trained the model on these, it would pick up on things like "flights that depart late tend to arrive late" and look really accurate but that's useless because we wouldn't know the actual departure time when a user is trying to decide between two flights. We'd be feeding the model information it would never have in the real scenario and therefore it's predictions will be wrong.

The columns in `drop_cols` like `Reporting_Airline`, `OriginCityName`, `DistanceGroup`, and `DepTimeBlk` are just duplicates of information we already have in other columns. `Origin` was also dropped since every single row is SEA and therefore there's not really a pattern for the model to uncover with regards to predicting ArrDelay as both a delayed flight and an ontime flight have origin as SEA.

In [39]:
cutoff = pd.Timestamp('2025-10-01')#this is the cutoff because it correlates exactly with Q4 and also because it leaves plenty of data for training like about 88% for training and 12% for testing so that theres enough data to train on.
df['DepHour'] = df['CRSDepTime'] // 100
train_pool = df[df['FlightDate'] < cutoff] #gets all the rows to train on that are before this particular date and it also happens to be about 88% of the data 
test = df[df['FlightDate'] >= cutoff] # gets all the rows to test on that are after or on this date which also happens to be about 12% of the data

dates = np.sort(train_pool['FlightDate'].unique())# sorts the dates from training pool to unique dates basically gives you all the unique dates
for fold, train_fold, val_fold, train_dates, val_dates in make_cv_folds(train_pool, dates):
    print(fold, train_fold.shape[0], val_fold.shape[0], train_dates.min(), train_dates.max(), val_dates.min(), val_dates.max())


print(len(train_pool), train_pool['FlightDate'].nunique(), train_pool['FlightDate'].min(), train_pool['FlightDate'].max())
print(len(test), test['FlightDate'].nunique(), test['FlightDate'].min(), test['FlightDate'].max())
assert train_pool['FlightDate'].max() < test['FlightDate'].min()
assert len(train_pool) + len(test) == len(df)


0 41901 51554 2024-01-01T00:00:00.000000000 2024-04-18T00:00:00.000000000 2024-04-19T00:00:00.000000000 2024-08-02T00:00:00.000000000
1 93455 49684 2024-01-01T00:00:00.000000000 2024-08-02T00:00:00.000000000 2024-08-03T00:00:00.000000000 2024-11-16T00:00:00.000000000
2 143139 41431 2024-01-01T00:00:00.000000000 2024-11-16T00:00:00.000000000 2024-11-17T00:00:00.000000000 2025-03-02T00:00:00.000000000
3 184570 47184 2024-01-01T00:00:00.000000000 2025-03-02T00:00:00.000000000 2025-03-03T00:00:00.000000000 2025-06-16T00:00:00.000000000
4 231754 53895 2024-01-01T00:00:00.000000000 2025-06-16T00:00:00.000000000 2025-06-17T00:00:00.000000000 2025-09-30T00:00:00.000000000
285649 639 2024-01-01 00:00:00 2025-09-30 00:00:00
38841 92 2025-10-01 00:00:00 2025-12-31 00:00:00


## Train pool / test split and CV strategy (Issue 2)

Single cutoff date: `2025-10-01`. Everything before it is the **train pool**
(2024-01-01 to 2025-09-30, 639 dates, 285,649 rows); everything on or after it is
**test** (2025-10-01 to 2025-12-31, 92 dates, 38,841 rows, ~12% of the data).
Test is held out untouched until Issue 10 — it is never used to compare or select
models.

There is no separate fixed validation split. Every candidate model in Issues 3, 4,
5, 8, 9 is scored across the same 5 `TimeSeriesSplit` folds over the train pool, so
every comparison uses identical folds.

`TimeSeriesSplit` is run over the array of the train pool's *unique dates*, not over
rows, because flight rows are not equally spaced (242-555 flights/day) while calendar
dates are. Each fold's row membership is recovered afterward with
`train_pool['FlightDate'].isin(fold_dates)`.

**`gap=0`** (the default) is a deliberate choice, not an oversight: no feature in this
project is lagged or rolling, every predictor is known at booking time, so adjacent
dates share no constructed value that a zero gap could leak across a fold boundary.
Revisit this if a lagged or rolling feature is ever added.

**Seasonal limitation:** because the split is temporal, the test partition falls
entirely in Q4 (Oct-Dec). The Issue 10 holdout score therefore reflects winter
operations only, not a full-year average. `Month` and `DayOfWeek` stay in the feature
set for this reason, and this caveat should be repeated at Issues 6 and 15.


In [40]:
profile_cols = ['IATA_CODE_Reporting_Airline', 'Flight_Number_Reporting_Airline', 'Dest'] # Grouping by
for fold, train_fold, val_fold, train_dates, val_dates in make_cv_folds(train_pool, dates): # Calling the function to give the fold no, the rows of the train fold, the val fold, the dates that the train fold has, the val dates
    profile_median = train_fold.groupby(profile_cols)['ArrDelay'].median() # Grouping the rows in the train fold by the profile_cols and then finding the median for each group
    val_fold = val_fold.merge(profile_median.rename('pred').reset_index(), on=profile_cols, how='left') # This merges the profile_median, it first renames the ArrDelay column to pred to reduce confusion, then resets the index to turn the MultiIndex series into a small dataframe. It merges on the columns in profile_median, and if no match is found for that specific row then pred just returns NaN
    print(fold, val_fold['pred'].isna().sum(), len(val_fold)) # This basically talks about per fold, how many predictions had no match in the group and the len of val_fold is also there to see how much of the total rows does that number make

    # So 32% of the val rows in fold zero have no match at all amongst the groups that have been made, the plan is to not have them rely on per-profile median here because each group might only have 1-2 rows and the median from those rows is not really a good prediction since an outlier in such a small amount of rows can skew the median forward and therefore the predictions will be worse. The threshold is that each profile should have at least 10 rows for it to make a prediction using the per-profile grouping


    

0 16924 51554
1 9959 49684
2 6072 41431
3 11989 47184
4 5033 53895


fold 0 showed 32% of the val rows had no match at all in the training rows for that fold, but that number is actually lower than the real problem because right now the median is being computed for every profile that shows up even once, so profiles with just 1 or 2 rows are still getting a median instead of counting as "no match". the real number of rows that need a fallback is probably higher than 32% once a threshold gets added.

what the threshold does is for every row it just checks two things - is there a profile for it at all in the training data, and does that profile have enough rows (>=10) to trust the median. if both are true it uses the median from training, if not it falls back to the coarser grouping. thats it.


n=10 was picked by checking fold 0, the thinnest fold, since that's where the choice matters most. At n=10, only 5.0% of that fold's training rows sit in profiles too thin to use, even though 39.9% of individual profiles fall below it - most thin profiles just don't carry much row weight, so raising the bar to 10 doesn't throw away much data. n=5 barely changes that (1.2% excluded) but trusts a median computed from as few as 5-9 points, which is risky given ArrDelay's fat right tail (p99=154, max=3359) - a single outlier flight could swing a median that small. n=15 and n=20 roughly double the exclusion rate for no real gain in stability over n=10. This threshold is applied the same way at every rung of the ladder and kept fixed across all 5 folds.


In [41]:
# Thresholding approach
mae_scores =[] # list to calculate MAE scores per fold
ladder_mae=[] #this is basically to calculate mae per fold of the rows that rung 1 was not able to predict because of not enough values in the profile or that there was no profile like that.They fell on to rung 2 or rung 3
flat_mae =[] # same as above but what this does is that it doesnt go through rung 2 it just replaces the prediciton for rows that rung 1 didnt predict with rung 3 which is the global median.This list stores the mae per fold in that case in order to compare it to the ladder mae
for fold, train_fold, val_fold, train_dates, val_dates in make_cv_folds(train_pool, dates):
    profile_stats = train_fold.groupby(profile_cols)['ArrDelay'].agg(['median', 'count'])
    reliable_profiles = profile_stats[profile_stats['count'] >= 10]['median'] # Gives the reliable profiles, the ones that have at least ten rows, and once they are identified their medians are taken. The code works by first giving a boolean mask for profiles with at least 10 rows, and then another filter on profile_stats keeps only those rows, after which their median is taken and count is discarded to keep only the reliable profiles
    val_fold = val_fold.merge(reliable_profiles.rename('pred').reset_index(), on=profile_cols, how='left') # Just does the merge with the validation fold, same as above
    val_fold['rung'] = np.where(val_fold['pred'].notna(),1,np.nan) # This basically creates a new column called rung to track which pred values are filled by which rung, since this is the first merge all the pred values come from rung 1, and this marks them as so in the rung column
    # print(fold, val_fold['pred'].isna().sum(), len(val_fold))

    # Fallback rate increased as predicted because the thin profiles are now not being considered, and if they are not being considered then there are more NaN's which increased the fallback rate

    coarse_stats = train_fold.groupby(['IATA_CODE_Reporting_Airline', 'DepHour'])['ArrDelay'].agg(['median', 'count'])
    reliable_coarse = coarse_stats[coarse_stats['count'] >= 10]['median'] # 10 is the count here as well, to ensure that only the profiles with at least 10 rows have the median sent as a prediction
    val_fold = val_fold.merge(reliable_coarse.rename('pred_coarse').reset_index(), on=['IATA_CODE_Reporting_Airline', 'DepHour'], how='left')
    unresolved = val_fold['pred'].isna() # Creates a snapshot of the pred column to find out rows which have the value of NaN
    val_fold['pred'] = val_fold['pred'].fillna(val_fold['pred_coarse']) # Replaces the prediction with the coarser prediction when pred is NaN, which means there was no match found in the most specific grouping, and replaces it with the prediction from a less specific grouping
    val_fold.loc[unresolved & val_fold['pred'].notna(),'rung'] = 2 # This line basically, what it does is, it finds the rows that were not resolved by rung 1 and does an elementwise AND with rows that are now resolved by both rung1 and rung2, and due to this elementwise AND, the only rows it shows true on are rows that were unresolved by the first rung which are now resolved by the second rung. The .loc[] then appropriately puts the value in the rung column as 2
    # print(fold, val_fold['pred'].isna().sum(), len(val_fold))

    still_unresolved = val_fold['pred'].isna() # Finds rows that are still unresolved after rung 1 and rung 2
    global_median = train_fold['ArrDelay'].median() # Calculate global median for rung 3
    val_fold['pred'] = val_fold['pred'].fillna(global_median) # Fill the NaN values with the global median, the first two rungs ensure that this is done for the least amount of rows as much as possible
    val_fold.loc[still_unresolved, 'rung'] = 3 # This labels the rows that were still unresolved after rung 1 and rung 2, and assigns them the value of 3.
    # print(fold,val_fold['pred'].isna().sum())
    non_rung_1 = val_fold[val_fold['rung'] != 1] #gets all the rows that were not predicited with rung 1
    fold_ladder_mae = (non_rung_1['ArrDelay'] - non_rung_1['pred']).abs().mean() # computes the mae for rows that were not predicted with rung 1 and rung 2
    ladder_mae.append(fold_ladder_mae)
    
    fold_flat_mae = (non_rung_1['ArrDelay']- global_median).abs().mean() #calculates mae on the rows that were not predicted with rung 1 but basically uses global median as the prediction for those rows and calculates the mae accordingly
    flat_mae.append(fold_flat_mae) 
    fold_mae = (val_fold['ArrDelay'] - val_fold['pred']).abs().mean() # calculating MAE for a single fold 
    mae_scores.append(fold_mae) # add ing the MAE for a particular fold to the mae_scores list
    print(fold, val_fold['rung'].value_counts().sort_index().to_dict()) # Gives the breakdown of how many values have been influenced by which rungs

print(np.mean(mae_scores), np.std(mae_scores)) #calculates mean to get the typical error of this approach and std tells how much those numbers differ amongst seperate folds.

#Mean MAE is 20 minutes for this approach and std is small at 1.2 which means theres not much variation in the performance as seasons/time passes

print(np.mean(ladder_mae),np.std(ladder_mae)) # prints out the mean across all 5 folds and the standard deviation to see performance across all five folds
print(np.mean(flat_mae),np.std(flat_mae))# same as above but in the case of when all the non-rung 1 fall straight to rung 3
print(np.mean(flat_mae) - np.mean(ladder_mae)) #tests to see if rung 2 is actually helping compared to just falling back to rung 3

0 {1.0: 30283, 2.0: 20641, 3.0: 630}
1 {1.0: 39274, 2.0: 10143, 3.0: 267}
2 {1.0: 34109, 2.0: 7308, 3.0: 14}
3 {1.0: 34643, 2.0: 12371, 3.0: 170}
4 {1.0: 39394, 2.0: 14356, 3.0: 145}
19.951910939426405 1.209756490038425
20.42569170229878 1.433145591603604
20.84019793146333 1.3210408121534298
0.41450622916454805


The first rung 2 grouped by carrier and destination and only got a lift of 0.149 minutes over just falling straight to the flat global median, basically nothing since both were scoring around 20.7-20.8 minutes MAE anyway.

Instead of guessing a better grouping, 11 different groupings got tested on the same 5 folds - carrier alone, Dest alone, departure hour alone, and combos of carrier/Dest/departure hour/month/day of week. Carrier + DepHour won clearly and roughly tripled the lift to 0.415 minutes. Two things came out of that search. Destination actually hurts - carrier alone (lift 0.210) beat carrier + Dest (0.149), and carrier + Dest + month dropped to basically -0.001, which is no better than the flat median at all, so adding destination isn't adding signal, it's just splitting groups into smaller, noisier ones. Departure hour on the other hand does carry a real effect Dest doesn't - the median ArrDelay by scheduled departure hour on the train pool shows 6am flights running a median of -10 minutes while 11am to 9pm flights sit near 0, so that's about a 9-10 minute swing across the day, which is bigger than any route level difference, probably because of delay propagation, where a delay earlier in an aircraft's day compounds through its later legs so a flight scheduled later inherits more of the day's accumulated slippage. DepHour is just a groupby key here, not a model feature, so the HHMM midnight wraparound problem doesn't apply, since group labels don't carry any ordering - hour 23 and hour 0 being far apart numerically doesn't matter here.

Also checked how good any median based rung could realistically get. Predicting the flat global median for every row gets 20.337 MAE, and a cheating oracle that uses each profile's true median, computed directly on the validation rows it's predicting (not something available in reality, just a ceiling check), gets 18.636. So the entire gap available to any median per group baseline is only about 1.70 minutes, because ArrDelay's spread around its own median is huge (mean absolute deviation ~20.4 minutes, p10=-22, p90=+36) compared to the difference between any two groups' centers. The current ladder is already close to that ceiling. Worth keeping in mind for Issue 6 - a model that can't beat this baseline by much isn't necessarily broken, since the naive per-profile median is already close to the best a median only approach can do.

One caveat though, the winning grouping was picked by scoring 11 candidates on the same 5 TimeSeriesSplit folds used everywhere else in this notebook, which is a mild form of selection on the CV folds. Defensible here because a stronger baseline makes the later "does the model actually beat the baseline" comparison harder, not easier, but this should be said plainly in Issue 6 instead of left implicit.
